# Single Persona Inference

This snippet will attempt the questionnaire for a single persona, record results and then dump them in a csv file for further downstream analysis!

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import pandas as pd
from tqdm import tqdm

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained("aryashah00/survey-finetuned-TinyLlama-1.1B-Chat-v1.0", device_map="auto", trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained("aryashah00/survey-finetuned-TinyLlama-1.1B-Chat-v1.0", trust_remote_code=True)

# Define WHOQOL-BREF questions
whoqol_questions = [
    "How would you rate your quality of life?",
    "How satisfied are you with your health?",
    "To what extent do you feel that physical pain prevents you from doing what you need to do?",
    "How much do you need any medical treatment to function in your daily life?",
    "How much do you enjoy life?",
    "To what extent do you feel your life to be meaningful?",
    "How well are you able to concentrate?",
    "How safe do you feel in your daily life?",
    "How healthy is your physical environment?",
    "Do you have enough energy for everyday life?",
    "Are you able to accept your bodily appearance?",
    "Have you enough money to meet your needs?",
    "How available to you is the information that you need in your day-to-day life?",
    "To what extent do you have the opportunity for leisure activities?",
    "How well are you able to get around?",
    "How satisfied are you with your sleep?",
    "How satisfied are you with your ability to perform your daily living activities?",
    "How satisfied are you with your capacity for work?",
    "How satisfied are you with yourself?",
    "How satisfied are you with your personal relationships?",
    "How satisfied are you with your sex life?",
    "How satisfied are you with the support you get from your friends?",
    "How satisfied are you with the conditions of your living place?",
    "How satisfied are you with your access to health services?",
    "How satisfied are you with your transport?",
    "How often do you have negative feelings such as blue mood, despair, anxiety, depression?"
]

# Define multiple personas
personas = [

    "A 55-year-old individual recently diagnosed with a chronic illness"
]

def generate_response(persona, question):
    """Generate a response for a given persona and question"""
    # Prepare prompts
    system_prompt = f"You are embodying the following persona: {persona}"
    user_prompt = f"WHOQOL-BREF Survey Question: {question}\n\nPlease provide your honest response to this question as if you were this person. Rate on a scale of 1-5 where appropriate (1=very poor/very dissatisfied/not at all, 5=very good/very satisfied/completely) and explain your rating."

    # Create message format
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    # Apply chat template
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # Tokenize
    input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to(model.device)

    # Generate response
    with torch.no_grad():
        output_ids = model.generate(
            input_ids=input_ids,
            max_new_tokens=256,
            temperature=0.7,
            top_p=0.9,
            do_sample=True
        )

    # Decode
    output = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    # Extract just the generated response
    response_start = output.find(input_text) + len(input_text)
    generated_response = output[response_start:].strip()

    return generated_response

# Create a dataframe to store all responses
results = []

# Generate responses for each persona and question
for persona in tqdm(personas, desc="Processing personas"):
    for question in tqdm(whoqol_questions, desc=f"Questions for {persona[:20]}...", leave=False):
        response = generate_response(persona, question)

        # Extract numerical rating if present (simple regex could be added here)
        # For now, we'll just store the full response

        results.append({
            "Persona": persona,
            "Question": question,
            "Response": response
        })

# Convert to DataFrame
results_df = pd.DataFrame(results)

# Save results
results_df.to_csv("whoqol_bref_persona_responses.csv", index=False)
print(f"Generated {len(results)} responses and saved to whoqol_bref_persona_responses.csv")


# Example of how you might analyze the results for each persona
persona_analyses = []
for persona in personas:
    persona_responses = results_df[results_df["Persona"] == persona]

print("Analysis complete.")


Processing personas: 100%|██████████| 1/1 [02:31<00:00, 151.07s/it]

Generated 26 responses and saved to whoqol_bref_persona_responses.csv
Analysis complete.


# Sub Agents: Multiple Personas attempting the same questionnaire

This time more than one persona attmepts the questioannire, each persona is a sub agent of the parent tinyllama model. Again, their results will be dumped in a csv for further downstream analysis or research

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import pandas as pd
from tqdm import tqdm

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained("aryashah00/survey-finetuned-TinyLlama-1.1B-Chat-v1.0", device_map="auto", trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained("aryashah00/survey-finetuned-TinyLlama-1.1B-Chat-v1.0", trust_remote_code=True)

# Define WHOQOL-BREF questions
whoqol_questions = [
    "How would you rate your quality of life?",
    "How satisfied are you with your health?",
    "To what extent do you feel that physical pain prevents you from doing what you need to do?",
    "How much do you need any medical treatment to function in your daily life?",
    "How much do you enjoy life?",
    "To what extent do you feel your life to be meaningful?",
    "How well are you able to concentrate?",
    "How safe do you feel in your daily life?",
    "How healthy is your physical environment?",
    "Do you have enough energy for everyday life?",
    "Are you able to accept your bodily appearance?",
    "Have you enough money to meet your needs?",
    "How available to you is the information that you need in your day-to-day life?",
    "To what extent do you have the opportunity for leisure activities?",
    "How well are you able to get around?",
    "How satisfied are you with your sleep?",
    "How satisfied are you with your ability to perform your daily living activities?",
    "How satisfied are you with your capacity for work?",
    "How satisfied are you with yourself?",
    "How satisfied are you with your personal relationships?",
    "How satisfied are you with your sex life?",
    "How satisfied are you with the support you get from your friends?",
    "How satisfied are you with the conditions of your living place?",
    "How satisfied are you with your access to health services?",
    "How satisfied are you with your transport?",
    "How often do you have negative feelings such as blue mood, despair, anxiety, depression?"
]

# Define multiple personas
personas = [
    "A 35-year-old software engineer with chronic back pain who works remotely",
    "A 68-year-old retiree who recently lost their spouse and lives alone",
    "A 22-year-old college student who is active in sports and social activities",
    "A 42-year-old single parent working two jobs to support their family",
    "A 55-year-old individual recently diagnosed with a chronic illness"
]

def generate_response(persona, question):
    """Generate a response for a given persona and question"""
    # Prepare prompts
    system_prompt = f"You are embodying the following persona: {persona}"
    user_prompt = f"WHOQOL-BREF Survey Question: {question}\n\nPlease provide your honest response to this question as if you were this person. Rate on a scale of 1-5 where appropriate (1=very poor/very dissatisfied/not at all, 5=very good/very satisfied/completely) and explain your rating."

    # Create message format
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    # Apply chat template
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # Tokenize
    input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to(model.device)

    # Generate response
    with torch.no_grad():
        output_ids = model.generate(
            input_ids=input_ids,
            max_new_tokens=256,
            temperature=0.7,
            top_p=0.9,
            do_sample=True
        )

    # Decode
    output = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    # Extract just the generated response
    response_start = output.find(input_text) + len(input_text)
    generated_response = output[response_start:].strip()

    return generated_response

# Create a dataframe to store all responses
results = []

# Generate responses for each persona and question
for persona in tqdm(personas, desc="Processing personas"):
    for question in tqdm(whoqol_questions, desc=f"Questions for {persona[:20]}...", leave=False):
        response = generate_response(persona, question)

        # Extract numerical rating if present (simple regex could be added here)
        # For now, we'll just store the full response

        results.append({
            "Persona": persona,
            "Question": question,
            "Response": response
        })

# Convert to DataFrame
results_df = pd.DataFrame(results)

# Save results
results_df.to_csv("whoqol_bref_persona_responses.csv", index=False)
print(f"Generated {len(results)} responses and saved to whoqol_bref_persona_responses.csv")

# Example of how you might analyze the results for each persona
persona_analyses = []
for persona in personas:
    persona_responses = results_df[results_df["Persona"] == persona]

print("Analysis complete.")


# For Swaraj: Things to lookout for:
1. Generalizing the script such that it answers the uploaded questionnaire by the user, i am guessing all you gotta do is replace the "questions" above by passing the list of the questionas once parsed from the user who uploaded it
2. CSV Dumping logic: basically extracting the responses(can vary with questionnaire types)
3. other nuances that might come up although i dont see any huge roadbloacks, not even minor!